In [2]:
# === FOLDER STRUCTURE DIAGNOSTIC ===
import os

DATASET_ROOT = r"f:\Research\Project\Final\infant-growth-monitoring-system\mlModels\CryTranslater\data\raw\img\Combined dataset"

print("🔍 CHECKING FOLDER STRUCTURE")
print("="*60)
print(f"Root path: {DATASET_ROOT}")
print(f"Root exists: {os.path.exists(DATASET_ROOT)}")

if os.path.exists(DATASET_ROOT):
    print(f"\n📁 Contents of root folder:")
    contents = os.listdir(DATASET_ROOT)
    for item in contents:
        item_path = os.path.join(DATASET_ROOT, item)
        is_dir = os.path.isdir(item_path)
        if is_dir:
            image_count = len([f for f in os.listdir(item_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            print(f"  📂 {item}/ ({image_count} images)")
        else:
            print(f"  📄 {item}")
else:
    print("❌ Root folder does not exist!")

print("\n💡 Expected folder structure:")
print("  Combined dataset/")
print("    ├── Pain/")
print("    └── No_Pain/")
print("\n(Note: Folder names are case-sensitive!)")

🔍 CHECKING FOLDER STRUCTURE
Root path: f:\Research\Project\Final\infant-growth-monitoring-system\mlModels\CryTranslater\data\raw\img\Combined dataset
Root exists: True

📁 Contents of root folder:
  📂 No_Pain/ (2135 images)
  📂 Pain/ (6368 images)

💡 Expected folder structure:
  Combined dataset/
    ├── Pain/
    └── No_Pain/

(Note: Folder names are case-sensitive!)


In [3]:
import os
import cv2
import sys
import numpy as np
import pandas as pd

# === 1. SAFE IMPORT & CONFIGURATION ===
try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
except ImportError as e:
    print(f"❌ MediaPipe not installed properly. Error: {e}")
    print("Run: pip install mediapipe")
    sys.exit()

# Your specific paths
DATASET_ROOT = r"f:\Research\Project\Final\infant-growth-monitoring-system\mlModels\CryTranslater\data\raw\img\Combined dataset"
OUTPUT_FILE = r"f:\Research\Project\Final\infant-growth-monitoring-system\mlModels\CryTranslater\data\processed\img_biomarkers3.csv"

# === 2. MEDIAPIPE SETUP (Using new API) ===
# We'll initialize the detector inside the processing function

# === 3. LANDMARK INDICES (Standard MediaPipe Topology) ===
# Eye Indices (Top, Bottom, Left, Right)
LEFT_EYE = [159, 145, 133, 33]
RIGHT_EYE = [386, 374, 362, 263]

# Mouth Indices (Top, Bottom, Left, Right)
MOUTH = [13, 14, 61, 291]

# Brow Indices for AU4 (Brow Lowering)
# We measure the distance from Inner Brow to Inner Eye Corner
LEFT_BROW_INNER = 55
LEFT_EYE_INNER = 133
RIGHT_BROW_INNER = 285
RIGHT_EYE_INNER = 362

def calculate_ratio(landmarks, indices):
    """
    Calculates Aspect Ratio: Vertical Dist / Horizontal Dist
    """
    # Get coordinates
    top = np.array([landmarks[indices[0]].x, landmarks[indices[0]].y])
    bottom = np.array([landmarks[indices[1]].x, landmarks[indices[1]].y])
    left = np.array([landmarks[indices[2]].x, landmarks[indices[2]].y])
    right = np.array([landmarks[indices[3]].x, landmarks[indices[3]].y])

    # Euclidean Distances
    vertical_dist = np.linalg.norm(top - bottom)
    horizontal_dist = np.linalg.norm(left - right)

    # Avoid division by zero
    if horizontal_dist == 0:
        return 0.0
    
    return vertical_dist / horizontal_dist

def calculate_brow_score(landmarks):
    """
    Calculates normalized brow distance. 
    Lower values = Brows lowered (Pain/Frowning).
    """
    # 1. Get coordinates for Left Side
    l_brow = np.array([landmarks[LEFT_BROW_INNER].x, landmarks[LEFT_BROW_INNER].y])
    l_eye = np.array([landmarks[LEFT_EYE_INNER].x, landmarks[LEFT_EYE_INNER].y])
    
    # 2. Get coordinates for Right Side
    r_brow = np.array([landmarks[RIGHT_BROW_INNER].x, landmarks[RIGHT_BROW_INNER].y])
    r_eye = np.array([landmarks[RIGHT_EYE_INNER].x, landmarks[RIGHT_EYE_INNER].y])

    # 3. Calculate Distances
    l_dist = np.linalg.norm(l_brow - l_eye)
    r_dist = np.linalg.norm(r_brow - r_eye)

    # 4. Normalize by Inter-ocular distance (distance between eyes) to account for zoom
    # Landmarks 33 (Left Eye Outer) and 263 (Right Eye Outer)
    eye_span = np.linalg.norm(
        np.array([landmarks[33].x, landmarks[33].y]) - 
        np.array([landmarks[263].x, landmarks[263].y])
    )
    
    if eye_span == 0: return 0
    
    return (l_dist + r_dist) / (2 * eye_span)

def process_dataset(root_path):
    data = []
    classes = ["Pain", "No_Pain"] # Folder names
    stats = {"Pain": {"total": 0, "processed": 0, "failed": 0},
             "No_Pain": {"total": 0, "processed": 0, "failed": 0}}
    
    print(f"🚀 Starting Feature Extraction from: {root_path}")
    print(f"📂 Root directory exists: {os.path.exists(root_path)}")
    
    # Download face landmarker model if not exists
    model_path = 'face_landmarker.task'
    if not os.path.exists(model_path):
        print("📥 Downloading face landmarker model...")
        import urllib.request
        url = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
        urllib.request.urlretrieve(url, model_path)
        print("✅ Model downloaded!")
    
    # Initialize detector with proper model path
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    detector = vision.FaceLandmarker.create_from_options(options)

    for class_name in classes:
        folder_path = os.path.join(root_path, class_name)
        # Label: 1 for Pain, 0 for No Pain
        label = 1 if class_name == "Pain" else 0
        
        print(f"\n{'='*60}")
        print(f"📁 Processing class: {class_name}")
        print(f"   Path: {folder_path}")
        print(f"   Exists: {os.path.exists(folder_path)}")
        
        if not os.path.exists(folder_path):
            print(f"⚠️ WARNING: Folder not found!")
            continue
            
        images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        stats[class_name]["total"] = len(images)
        print(f"   Images found: {len(images)}")

        for idx, img_name in enumerate(images):
            img_path = os.path.join(folder_path, img_name)
            
            try:
                # Load image
                image = cv2.imread(img_path)
                if image is None:
                    stats[class_name]["failed"] += 1
                    continue
                rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                
                # Convert to MediaPipe Image format
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
                
                # Run MediaPipe face landmarker
                detection_result = detector.detect(mp_image)

                if detection_result.face_landmarks:
                    lm = detection_result.face_landmarks[0]
                    
                    # --- FEATURE EXTRACTION ---
                    # 1. EAR (Eye Aspect Ratio) - Low = Pain/Sleep
                    left_ear = calculate_ratio(lm, LEFT_EYE)
                    right_ear = calculate_ratio(lm, RIGHT_EYE)
                    avg_ear = (left_ear + right_ear) / 2.0

                    # 2. MAR (Mouth Aspect Ratio) - High = Crying
                    mar = calculate_ratio(lm, MOUTH)

                    # 3. Brow Distance (AU4) - Low = Pain
                    brow_score = calculate_brow_score(lm)

                    data.append({
                        "filename": img_name,
                        "ear": avg_ear,
                        "mar": mar,
                        "brow_score": brow_score,
                        "label": label
                    })
                    stats[class_name]["processed"] += 1
                else:
                    stats[class_name]["failed"] += 1
                    
                # Progress indicator
                if (idx + 1) % 50 == 0:
                    print(f"   Progress: {idx + 1}/{len(images)} images")
                    
            except Exception as e:
                stats[class_name]["failed"] += 1
                if stats[class_name]["failed"] <= 3:  # Show first 3 errors
                    print(f"   ⚠️ Error processing {img_name}: {str(e)}")
                continue
    
    detector.close()
    
    # Print detailed statistics
    print(f"\n{'='*60}")
    print("📊 PROCESSING SUMMARY")
    print(f"{'='*60}")
    for class_name in classes:
        print(f"\n{class_name}:")
        print(f"  Total images: {stats[class_name]['total']}")
        print(f"  Successfully processed: {stats[class_name]['processed']}")
        print(f"  Failed/No face detected: {stats[class_name]['failed']}")
    
    return pd.DataFrame(data)

# === 4. EXECUTION ===
if __name__ == "__main__":
    # Ensure output directory exists
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    
    df = process_dataset(DATASET_ROOT)
    
    if not df.empty:
        # Show class distribution
        print(f"\n{'='*60}")
        print("📊 FINAL DATASET DISTRIBUTION")
        print(f"{'='*60}")
        print(f"Total samples: {len(df)}")
        print(f"\nClass distribution:")
        print(df['label'].value_counts().to_string())
        print(f"\nLabel 0 (No Pain): {(df['label'] == 0).sum()} samples")
        print(f"Label 1 (Pain): {(df['label'] == 1).sum()} samples")
        
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n✅ Extraction Complete!")
        print(f"💾 Saved to: {OUTPUT_FILE}")
        
        # Show quick stats to verify data quality
        print("\n📈 Feature Statistics by Class:")
        print(df.groupby('label')[['ear', 'mar', 'brow_score']].mean())
    else:
        print("❌ No data extracted. Check your folder paths and image files.")


🚀 Starting Feature Extraction from: f:\Research\Project\Final\infant-growth-monitoring-system\mlModels\CryTranslater\data\raw\img\Combined dataset
📂 Root directory exists: True

📁 Processing class: Pain
   Path: f:\Research\Project\Final\infant-growth-monitoring-system\mlModels\CryTranslater\data\raw\img\Combined dataset\Pain
   Exists: True
   Images found: 6368
   Progress: 50/6368 images
   Progress: 100/6368 images
   Progress: 150/6368 images
   Progress: 200/6368 images
   Progress: 250/6368 images
   Progress: 300/6368 images
   Progress: 350/6368 images
   Progress: 400/6368 images
   Progress: 450/6368 images
   Progress: 500/6368 images
   Progress: 550/6368 images
   Progress: 600/6368 images
   Progress: 650/6368 images
   Progress: 700/6368 images
   Progress: 750/6368 images
   Progress: 800/6368 images
   Progress: 850/6368 images
   Progress: 900/6368 images
   Progress: 950/6368 images
   Progress: 1000/6368 images
   Progress: 1050/6368 images
   Progress: 1100/6368 i